# Denoising Diffusion for Molecular Coordinate Generation

**Author:** Nandini  
**Stack:** PyTorch, NumPy, matplotlib  
**Topics:** DDPM fundamentals, noise schedules, denoising score matching, coordinate generation for molecular-like 2D point clouds

Simplified demonstration of diffusion model mechanics on 2D molecular conformations (ring structures).

---

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
torch.manual_seed(42)
np.random.seed(42)
print(f'PyTorch {torch.__version__}')

PyTorch 2.11.0+cu130


## 1. Data: 2D "Molecular" Conformations
Generate ring-like point cloud data resembling benzene/cyclohexane conformations with noise.

In [2]:
def generate_ring_conformations(n_samples=1000, n_atoms=6, radius=1.5, noise=0.08):
    """Generate 2D ring conformations (like benzene viewed from above)."""
    data = []
    for _ in range(n_samples):
        # Random rotation
        theta_offset = np.random.uniform(0, 2 * np.pi)
        r = radius + np.random.normal(0, 0.1)  # slight radius variation
        angles = np.linspace(0, 2 * np.pi, n_atoms, endpoint=False) + theta_offset
        coords = np.stack([r * np.cos(angles), r * np.sin(angles)], axis=1)
        coords += np.random.normal(0, noise, coords.shape)
        # Center
        coords -= coords.mean(axis=0)
        data.append(coords)
    return torch.tensor(np.array(data), dtype=torch.float32)

# Mix of 5-rings and 6-rings
data_6 = generate_ring_conformations(n_samples=500, n_atoms=6, radius=1.5)
data_5 = generate_ring_conformations(n_samples=500, n_atoms=5, radius=1.3)

# Use 6-membered rings for this demo
data = data_6  # [500, 6, 2]
data_flat = data.reshape(data.shape[0], -1)  # [500, 12]
print(f'Dataset: {data.shape} -> flattened {data_flat.shape}')

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    coords = data[i].numpy()
    # Draw bonds (ring)
    for j in range(len(coords)):
        k = (j + 1) % len(coords)
        axes[i].plot([coords[j, 0], coords[k, 0]], [coords[j, 1], coords[k, 1]], 'k-', alpha=0.3)
    axes[i].scatter(coords[:, 0], coords[:, 1], c='#3b82f6', s=80, zorder=3, edgecolors='white')
    axes[i].set_xlim(-2.5, 2.5)
    axes[i].set_ylim(-2.5, 2.5)
    axes[i].set_aspect('equal')
    axes[i].set_title(f'Sample {i}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Training Data: 6-Membered Ring Conformations')
plt.tight_layout()
plt.show()

Dataset: torch.Size([500, 6, 2]) -> flattened torch.Size([500, 12])


## 2. DDPM Forward Process

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\, I)$$

In [3]:
T = 200  # diffusion steps
beta_start, beta_end = 1e-4, 0.02

# Linear schedule
betas = torch.linspace(beta_start, beta_end, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, noise=None):
    """Add noise at timestep t."""
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab = torch.sqrt(alpha_bar[t]).reshape(-1, 1)
    sqrt_1_ab = torch.sqrt(1 - alpha_bar[t]).reshape(-1, 1)
    return sqrt_ab * x0 + sqrt_1_ab * noise, noise

# Visualize forward process
sample = data_flat[0:1]  # single molecule
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
timesteps_viz = [0, 20, 50, 100, 150, 199]

for idx, t_val in enumerate(timesteps_viz):
    t_tensor = torch.tensor([t_val])
    noisy, _ = forward_diffusion(sample, t_tensor)
    coords = noisy.reshape(6, 2).numpy()
    for j in range(6):
        k = (j + 1) % 6
        axes[idx].plot([coords[j, 0], coords[k, 0]], [coords[j, 1], coords[k, 1]], 'k-', alpha=0.2)
    axes[idx].scatter(coords[:, 0], coords[:, 1], c='#ef4444', s=60, zorder=3)
    axes[idx].set_title(f't = {t_val}', fontsize=9)
    axes[idx].set_xlim(-3, 3)
    axes[idx].set_ylim(-3, 3)
    axes[idx].set_aspect('equal')
    axes[idx].axis('off')

plt.suptitle('Forward Diffusion: Ring Structure -> Noise')
plt.tight_layout()
plt.show()

## 3. Denoising Network
Simple MLP that predicts noise $\epsilon_\theta(x_t, t)$ given noisy coordinates and timestep.

In [4]:
class NoisePredictor(nn.Module):
    def __init__(self, data_dim=12, hidden=256, time_emb_dim=32):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_emb_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )
    
    def forward(self, x_t, t):
        t_emb = self.time_embed(t.float().unsqueeze(-1) / T)  # normalize t
        inp = torch.cat([x_t, t_emb], dim=-1)
        return self.net(inp)

denoiser = NoisePredictor()
n_params = sum(p.numel() for p in denoiser.parameters())
print(f'Denoiser parameters: {n_params}')

Denoiser parameters: 147308


## 4. Training: Denoising Score Matching

In [5]:
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
batch_size = 64
n_epochs = 200
losses = []

for epoch in range(n_epochs):
    perm = torch.randperm(len(data_flat))
    epoch_loss = 0
    n_batches = 0
    
    for i in range(0, len(data_flat), batch_size):
        batch = data_flat[perm[i:i+batch_size]]
        bs = batch.shape[0]
        
        # Sample random timesteps
        t = torch.randint(0, T, (bs,))
        
        # Forward diffusion
        noise = torch.randn_like(batch)
        x_t, _ = forward_diffusion(batch, t, noise)
        
        # Predict noise
        noise_pred = denoiser(x_t, t)
        loss = nn.functional.mse_loss(noise_pred, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 40 == 0:
        print(f'Epoch {epoch+1:4d} | Loss: {avg_loss:.6f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#8b5cf6')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Denoising Score Matching Training')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

Epoch   40 | Loss: 0.236489


Epoch   80 | Loss: 0.229386


Epoch  120 | Loss: 0.225313


Epoch  160 | Loss: 0.203706


Epoch  200 | Loss: 0.196183


## 5. Sampling: Reverse Diffusion
Generate new molecular conformations by denoising from pure Gaussian noise.

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t, t)\right) + \sigma_t z$$

In [6]:
@torch.no_grad()
def sample(model, n_samples=10, data_dim=12):
    model.eval()
    x = torch.randn(n_samples, data_dim)  # Start from pure noise
    
    trajectory = [x.clone()]
    
    for t_val in reversed(range(T)):
        t = torch.full((n_samples,), t_val, dtype=torch.long)
        
        # Predict noise
        eps_pred = model(x, t)
        
        # DDPM reverse step
        alpha_t = alphas[t_val]
        alpha_bar_t = alpha_bar[t_val]
        beta_t = betas[t_val]
        
        x = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * eps_pred)
        
        if t_val > 0:
            sigma = torch.sqrt(beta_t)
            x += sigma * torch.randn_like(x)
        
        if t_val % 40 == 0:
            trajectory.append(x.clone())
    
    return x, trajectory

generated, trajectory = sample(denoiser, n_samples=8)
print(f'Generated {generated.shape[0]} conformations')

Generated 8 conformations


In [7]:
# Visualize generated samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Row 1: Training data
for i in range(4):
    coords = data[i].numpy()
    for j in range(6):
        k = (j + 1) % 6
        axes[0, i].plot([coords[j, 0], coords[k, 0]], [coords[j, 1], coords[k, 1]], '#3b82f6', alpha=0.4)
    axes[0, i].scatter(coords[:, 0], coords[:, 1], c='#3b82f6', s=80, zorder=3, edgecolors='white')
    axes[0, i].set_title('Training' if i == 0 else '', fontsize=9)
    axes[0, i].set_xlim(-3, 3)
    axes[0, i].set_ylim(-3, 3)
    axes[0, i].set_aspect('equal')
    axes[0, i].axis('off')

# Row 2: Generated
for i in range(4):
    coords = generated[i].reshape(6, 2).numpy()
    for j in range(6):
        k = (j + 1) % 6
        axes[1, i].plot([coords[j, 0], coords[k, 0]], [coords[j, 1], coords[k, 1]], '#ef4444', alpha=0.4)
    axes[1, i].scatter(coords[:, 0], coords[:, 1], c='#ef4444', s=80, zorder=3, edgecolors='white')
    axes[1, i].set_title('Generated' if i == 0 else '', fontsize=9)
    axes[1, i].set_xlim(-3, 3)
    axes[1, i].set_ylim(-3, 3)
    axes[1, i].set_aspect('equal')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Training Data', fontsize=10)
axes[1, 0].set_ylabel('DDPM Generated', fontsize=10)
plt.suptitle('Diffusion Model: Training vs Generated Ring Conformations', fontsize=12)
plt.tight_layout()
plt.show()

In [8]:
# Denoising trajectory visualization
fig, axes = plt.subplots(1, len(trajectory), figsize=(3 * len(trajectory), 3))
t_labels = list(reversed(range(0, T, 40))) + [0]

for idx, snap in enumerate(trajectory):
    coords = snap[0].reshape(6, 2).numpy()
    for j in range(6):
        k = (j + 1) % 6
        axes[idx].plot([coords[j, 0], coords[k, 0]], [coords[j, 1], coords[k, 1]], 'k-', alpha=0.2)
    axes[idx].scatter(coords[:, 0], coords[:, 1], c='#8b5cf6', s=50, zorder=3)
    axes[idx].set_xlim(-3.5, 3.5)
    axes[idx].set_ylim(-3.5, 3.5)
    axes[idx].set_aspect('equal')
    axes[idx].axis('off')
    step_label = 'noise' if idx == 0 else ('final' if idx == len(trajectory)-1 else f'step {idx}')
    axes[idx].set_title(step_label, fontsize=8)

plt.suptitle('Reverse Diffusion Trajectory: Noise -> Ring Conformation')
plt.tight_layout()
plt.show()

print('Done. The diffusion model learns to denoise Gaussian noise into ring-like molecular conformations.')

Done. The diffusion model learns to denoise Gaussian noise into ring-like molecular conformations.
